In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Config & imports
# ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import gc
import joblib
import os
import warnings
import lightgbm as lgb                          # ← ADDED — required for callbacks
from pathlib import Path

from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             classification_report, roc_auc_score,
                             precision_recall_curve)
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────
PROJECT_ROOT = Path("/Users/shashi/retail-ai-project")
NB_DIR       = PROJECT_ROOT / "notebooks"
CSV_PATH     = PROJECT_ROOT / "data/raw/output/csv"
MODELS_DIR   = PROJECT_ROOT / "models"
PROCESSED    = PROJECT_ROOT / "data/processed/training"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# ── Date split ────────────────────────────────────────────────────
TRAIN_END   = "2025-06-30"
VALID_START = "2025-07-01"
VALID_END   = "2025-09-30"
TEST_START  = "2025-10-01"
TEST_END    = "2025-12-31"

# ── Tiered CSL ────────────────────────────────────────────────────
Z_MAP = {0.90: 1.2816, 0.95: 1.6449, 0.975: 1.96}
CSL_BY_TIER = {
    "Premium": 0.975,
    "High":    0.975,
    "Medium":  0.95,
    "Low":     0.95,
}

# ── Segments ──────────────────────────────────────────────────────
HI_TIERS = ["High", "Premium"]
LO_TIERS = ["Low", "Medium"]

# ── Feature lists ─────────────────────────────────────────────────
V1_FEATURES = [
    "is_promoted", "category", "foot_traffic_tier", "region",
    "store_format", "price_band",
    "dow", "month", "day", "weekofyear", "is_weekend",
    "lag_1", "lag_7", "lag_14", "lag_28",
    "roll_mean_7", "roll_std_7", "roll_mean_28", "roll_std_28",
]
V22_EXTRA = [
    "discount_pct", "demand_lift_factor",
    "promo_depth_x_flag", "promo_lift_x_flag",
    "lost_units_proxy", "stockout_flag",
]
FORECAST_FEATURES = V1_FEATURES + V22_EXTRA
CAT_COLS = ["category", "foot_traffic_tier", "region", "store_format", "price_band"]

print("=" * 55)
print("  DemandSense_v2 — Retail AI Pipeline")
print("=" * 55)
print(f"  Train master : {NB_DIR / 'train_master_full.csv'}")
print(f"  Models dir   : {MODELS_DIR}")
print(f"  Features     : {len(FORECAST_FEATURES)}")

  DemandSense_v2 — Retail AI Pipeline
  Train master : /Users/shashi/retail-ai-project/notebooks/train_master_full.csv
  Models dir   : /Users/shashi/retail-ai-project/models
  Features     : 25


In [17]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Load data
# train_master_full.csv = 4.58 GB, already feature-engineered
# Only load promotions + stockout for V2.2 features
# ─────────────────────────────────────────────────────────────────
print("[Load] Reading train_master_full.csv (4.58 GB — this takes ~2 min)...")
df = pd.read_csv(NB_DIR / "train_master_full.csv", parse_dates=["date"])
print(f"  Shape: {df.shape}")
print(f"  Date range: {df['date'].min()} → {df['date'].max()}")
print(f"  Columns: {list(df.columns)}")

# Load promotions for V2.2 features (small file — 439 KB)
print("\n[Load] Reading promotions.csv...")
promotions = pd.read_csv(
    CSV_PATH / "promotions.csv",
    parse_dates=["start_date", "end_date"]
)
print(f"  Promotions: {len(promotions):,} rows")

# Load stockout events for V2.2 features (24 MB)
print("\n[Load] Reading stockout_events.csv...")
stockouts = pd.read_csv(
    CSV_PATH / "stockout_events.csv",
    parse_dates=["stockout_date", "restock_date"]
)
print(f"  Stockouts: {len(stockouts):,} rows")

# Load stores and products for safety stock + replenishment
print("\n[Load] Reading reference tables...")
stores    = pd.read_csv(CSV_PATH / "stores.csv")
products  = pd.read_csv(CSV_PATH / "products.csv")
suppliers = pd.read_csv(CSV_PATH / "suppliers.csv")

# Drop redundant column — same as target, just renamed when saving predictions
if "actual_units" in df.columns:
    df = df.drop(columns=["actual_units"])


print("\n All tables loaded")

[Load] Reading train_master_full.csv (4.58 GB — this takes ~2 min)...
  Shape: (24898386, 25)
  Date range: 2024-01-29 00:00:00 → 2025-12-31 00:00:00
  Columns: ['store_id', 'sku_id', 'date', 'target', 'actual_units', 'revenue', 'is_promoted', 'category', 'foot_traffic_tier', 'region', 'store_format', 'price_band', 'dow', 'month', 'day', 'weekofyear', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_std_7', 'roll_mean_28', 'roll_std_28']

[Load] Reading promotions.csv...
  Promotions: 5,420 rows

[Load] Reading stockout_events.csv...
  Stockouts: 313,599 rows

[Load] Reading reference tables...

 All tables loaded


In [18]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Add V2.2 features
# CRITICAL FIX: handle store_id=NULL (all-store promotions)
# ─────────────────────────────────────────────────────────────────
print("[V2.2 Features] Adding promotion depth features...")

promo = promotions.copy()

# ── CRITICAL 3 FIX: handle NULL store_id (all-store promotions) ───
# promotions.csv has store_id=NULL when promo applies to ALL stores
# The original exact join on store_id dropped all of these rows
# causing only 1,434 non-zero discount_pct out of 24M rows
promo_store_specific = promo[promo["store_id"].notna()].copy()
promo_all_stores     = promo[promo["store_id"].isna()].copy()

print(f"  Store-specific promos : {len(promo_store_specific):,}")
print(f"  All-store promos      : {len(promo_all_stores):,}")

if len(promo_all_stores) > 0:
    all_store_ids = df["store_id"].unique()
    expanded_list = []
    for sid in all_store_ids:
        tmp = promo_all_stores.drop(columns=["store_id"]).copy()
        tmp["store_id"] = sid
        expanded_list.append(tmp)
    promo_all_expanded = pd.concat(expanded_list, ignore_index=True)
    promo = pd.concat([promo_store_specific, promo_all_expanded], ignore_index=True)
    print(f"  After expanding all-store promos: {len(promo):,} rows")
else:
    promo = promo_store_specific

# Continue with existing logic — now with all promos included
promo = promo.dropna(subset=["store_id", "sku_id", "start_date", "end_date"])
promo = promo[promo["end_date"] >= promo["start_date"]].copy()

promo["discount_pct"] = (
    pd.to_numeric(promo["discount_pct"], errors="coerce")
    .fillna(0.0).clip(0, 0.95)
)
promo["demand_lift_factor"] = (
    pd.to_numeric(promo["demand_lift_factor"], errors="coerce")
    .fillna(1.0).clip(0.5, 5.0)
)

# Expand date ranges to daily rows
promo["date"] = promo.apply(
    lambda r: pd.date_range(r["start_date"], r["end_date"], freq="D"), axis=1
)
promo = promo.explode("date", ignore_index=True)
promo = (
    promo.groupby(["store_id", "sku_id", "date"], as_index=False)
    .agg(
        discount_pct=("discount_pct", "max"),
        demand_lift_factor=("demand_lift_factor", "max")
    )
)
print(f"  Daily promo rows after explode: {len(promo):,}")

df = df.merge(promo, on=["store_id", "sku_id", "date"], how="left")
df["discount_pct"]       = pd.to_numeric(df["discount_pct"], errors="coerce").fillna(0.0)
df["demand_lift_factor"] = pd.to_numeric(df["demand_lift_factor"], errors="coerce").fillna(1.0)
df["promo_depth_x_flag"] = df["discount_pct"] * df["is_promoted"].astype(int)
df["promo_lift_x_flag"]  = df["demand_lift_factor"] * df["is_promoted"].astype(int)
print(f"  Non-zero discount_pct: {(df['discount_pct']>0).sum():,}  (was 1,434 before fix)")

# ── Stockout proxy feature ────────────────────────────────────────
print("[V2.2 Features] Adding stockout proxy feature...")

lost = (
    stockouts
    .groupby(["store_id", "sku_id", "stockout_date"], as_index=False)["estimated_lost_units"]
    .sum()
    .rename(columns={"stockout_date": "date", "estimated_lost_units": "lost_units_proxy"})
)
df = df.merge(lost, on=["store_id", "sku_id", "date"], how="left")
df["lost_units_proxy"] = (
    pd.to_numeric(df["lost_units_proxy"], errors="coerce").fillna(0.0)
)
df["lost_units_proxy"] = df["lost_units_proxy"].clip(
    0, df["lost_units_proxy"].quantile(0.99)
)
df["stockout_flag"] = (df["lost_units_proxy"] > 0).astype(int)

print(f"  Stockout proxy added. Stockout days: {df['stockout_flag'].sum():,}")
print(f"  Final df shape: {df.shape}")
print(f"  All V2.2 features present: {all(c in df.columns for c in V22_EXTRA)}")
print("\n V2.2 features complete")

[V2.2 Features] Adding promotion depth features...
  Store-specific promos : 3,143
  All-store promos      : 2,277
  After expanding all-store promos: 1,091,549 rows
  Daily promo rows after explode: 9,540,201
  Non-zero discount_pct: 526,318  (was 1,434 before fix)
[V2.2 Features] Adding stockout proxy feature...
  Stockout proxy added. Stockout days: 289,342
  Final df shape: (24898386, 30)
  All V2.2 features present: True

 V2.2 features complete


In [19]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Time-based split
# Exact same dates as Forecast.ipynb
# CRITICAL FIX: removed train.sample(frac=0.35) — use full data
# ─────────────────────────────────────────────────────────────────
print("[Split] Applying time-based train/val/test split...")

train = df[df["date"] <= TRAIN_END].copy()
valid = df[(df["date"] >= VALID_START) & (df["date"] <= VALID_END)].copy()
test  = df[(df["date"] >= TEST_START)  & (df["date"] <= TEST_END)].copy()

# Cast categoricals — exact from Cell [221]
for c in CAT_COLS:
    if c in train.columns:
        train[c] = train[c].astype("category")
        valid[c] = valid[c].astype("category")
        test[c]  = test[c].astype("category")

# Fill any NaN in features
for split in [train, valid, test]:
    for col in FORECAST_FEATURES:
        if col in split.columns and col not in CAT_COLS:
            split[col] = split[col].fillna(0)

print(f"  Train: {len(train):>12,} rows  ({train['date'].min()} → {train['date'].max()})")
print(f"  Valid: {len(valid):>12,} rows  ({valid['date'].min()} → {valid['date'].max()})")
print(f"  Test : {len(test):>12,} rows  ({test['date'].min()} → {test['date'].max()})")
print(f"\n   NO sampling applied — full training data used")
print(f"  Features: {len(FORECAST_FEATURES)}")

[Split] Applying time-based train/val/test split...
  Train:   18,355,832 rows  (2024-01-29 00:00:00 → 2025-06-30 00:00:00)
  Valid:    3,271,115 rows  (2025-07-01 00:00:00 → 2025-09-30 00:00:00)
  Test :    3,271,439 rows  (2025-10-01 00:00:00 → 2025-12-31 00:00:00)

   NO sampling applied — full training data used
  Features: 25


In [20]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Helper functions
# FIXES:
#   1. Added Xtr/ytr/Xva/yva/Xte assignments at start of function
#   2. p50 instantiation now BEFORE p50.fit()
#   3. early_stopping(50) callback on ALL 3 models (P50, P05, P95)
#   4. Removed callbacks=[] which was disabling early stopping
# ─────────────────────────────────────────────────────────────────

def fit_pred_segment(tr, va, te, tag):
    """Train P50 + P05 + P95 for one store tier segment."""
    # ── Define X/y splits first ───────────────────────────────────
    Xtr, ytr = tr[FORECAST_FEATURES], tr["target"]
    Xva, yva = va[FORECAST_FEATURES], va["target"]
    Xte      = te[FORECAST_FEATURES]

    # ── P50 (point forecast) ──────────────────────────────────────
    p50 = LGBMRegressor(
        objective="regression",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=300,
        subsample=0.8,
        colsample_bytree=0.8,
        force_col_wise=True,
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    )
    p50.fit(
        Xtr, ytr,
        eval_set=[(Xva, yva)],
        eval_metric="l1",
        categorical_feature=CAT_COLS,
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(200),
        ]
    )
    print(f"  [{tag}] P50 best iter: {p50.best_iteration_}")

    out = te[["store_id", "sku_id", "date", "target",
              "category", "foot_traffic_tier"]].copy()
    out["forecast_units"] = p50.predict(Xte).clip(0)
    del p50; gc.collect()

    # ── P05 (lower bound) ─────────────────────────────────────────
    q05 = LGBMRegressor(
        objective="quantile", alpha=0.05,
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        min_child_samples=300, subsample=0.8, colsample_bytree=0.8,
        force_col_wise=True, n_jobs=-1, random_state=42, verbose=-1
    )
    q05.fit(
        Xtr, ytr,
        eval_set=[(Xva, yva)],
        eval_metric="quantile",
        categorical_feature=CAT_COLS,
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(200),
        ]
    )
    out["lower_bound_90"] = q05.predict(Xte).clip(0)
    del q05; gc.collect()

    # ── P95 (upper bound) ─────────────────────────────────────────
    q95 = LGBMRegressor(
        objective="quantile", alpha=0.95,
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        min_child_samples=300, subsample=0.8, colsample_bytree=0.8,
        force_col_wise=True, n_jobs=-1, random_state=42, verbose=-1
    )
    q95.fit(
        Xtr, ytr,
        eval_set=[(Xva, yva)],
        eval_metric="quantile",
        categorical_feature=CAT_COLS,
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(200),
        ]
    )
    out["upper_bound_90"] = q95.predict(Xte).clip(0)
    del q95; gc.collect()

    # ── Enforce lower ≤ upper (from Forecast.ipynb Cell [108]) ────
    swap = out["lower_bound_90"] > out["upper_bound_90"]
    out.loc[swap, ["lower_bound_90", "upper_bound_90"]] = \
        out.loc[swap, ["upper_bound_90", "lower_bound_90"]].values

    out["forecast_method"] = tag
    return out


def compute_metrics(pred, tag):
    """Exact metrics() from Forecast.ipynb Cell [221]."""
    y       = pred["target"]
    f       = pred["forecast_units"]
    err     = y - f
    abs_err = err.abs()
    ape     = abs_err / y.replace(0, np.nan) * 100
    cov     = ((y >= pred["lower_bound_90"]) & (y <= pred["upper_bound_90"])).mean() * 100
    return {
        "forecast_method": tag,
        "mape":         float(np.nanmean(ape)),
        "wape":         float(abs_err.sum() / y.sum() * 100),
        "mae":          float(mean_absolute_error(y, f)),
        "rmse":         float(np.sqrt(mean_squared_error(y, f))),
        "bias":         float(err.mean()),
        "p90_coverage": float(cov),
    }


def bev_snack_wape(pred, tag):
    """Category guard — Beverages + Snacks only.
    observed=True prevents NaN rows for other categories."""
    d = pred[pred["category"].isin(["Beverages", "Snacks"])].copy()
    d["abs_error"] = (d["target"] - d["forecast_units"]).abs()
    o = (d.groupby("category", as_index=False, observed=True)
          .agg(actual_sum=("target", "sum"), abs_err=("abs_error", "sum")))
    o["wape"]  = o["abs_err"] / o["actual_sum"] * 100
    o["model"] = tag
    return o[["model", "category", "wape"]]


print(" Helper functions defined")

 Helper functions defined


In [21]:
# ─────────────────────────────────────────────────────────────────
# CELL 6 — Train DemandSense_v2 (V2.2 segmented)
# Exact architecture from Forecast.ipynb Cell [221]
# Fix: removed train.sample(frac=0.35) — full training data
# ─────────────────────────────────────────────────────────────────
print("[Training] DemandSense_v2 — segmented LightGBM")
print(f"  Total train rows: {len(train):,}  ← NO SAMPLING")
print(f"  Features: {len(FORECAST_FEATURES)}")
print()

# High/Premium segment
print("── Segment: High + Premium stores ──")
pred_hi = fit_pred_segment(
    train[train["foot_traffic_tier"].isin(HI_TIERS)].copy(),
    valid[valid["foot_traffic_tier"].isin(HI_TIERS)].copy(),
    test[test["foot_traffic_tier"].isin(HI_TIERS)].copy(),
    "DemandSense_v2_HighPremium"
)

# Low/Medium segment
print("\n── Segment: Low + Medium stores ──")
pred_lo = fit_pred_segment(
    train[train["foot_traffic_tier"].isin(LO_TIERS)].copy(),
    valid[valid["foot_traffic_tier"].isin(LO_TIERS)].copy(),
    test[test["foot_traffic_tier"].isin(LO_TIERS)].copy(),
    "DemandSense_v2_LowMedium"
)

# Combine
pred_all = pd.concat([pred_hi, pred_lo], ignore_index=True)
print(f"\n  Total test predictions: {len(pred_all):,}")
print("\n DemandSense_v2 training complete")

[Training] DemandSense_v2 — segmented LightGBM
  Total train rows: 18,355,832  ← NO SAMPLING
  Features: 25

── Segment: High + Premium stores ──
[200]	valid_0's l1: 3.00097	valid_0's l2: 16.3222
  [DemandSense_v2_HighPremium] P50 best iter: 212
[200]	valid_0's quantile: 0.356475
[200]	valid_0's quantile: 0.41359

── Segment: Low + Medium stores ──
  [DemandSense_v2_LowMedium] P50 best iter: 136
[200]	valid_0's quantile: 0.217647
[200]	valid_0's quantile: 0.287853

  Total test predictions: 3,271,439

 DemandSense_v2 training complete


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 7 — Metrics + Champion/Challenger
# Exact logic from Forecast.ipynb Cell [221]
# Reference V1 metrics hardcoded from summary_v2_2_vs_v1.csv
# ─────────────────────────────────────────────────────────────────
print("[Evaluation] Computing metrics on Q4 test set...")

m_v2 = compute_metrics(pred_all, "DemandSense_v2")
bs_v2 = bev_snack_wape(pred_all, "DemandSense_v2")

# V1 reference metrics — from summary_v2_2_vs_v1.csv
v1_ref = {
    "forecast_method": "DemandSense_v1",
    "mape": 40.530127,
    "wape": 24.176468,
    "mae":  2.795133,
    "rmse": 4.027924,
    "bias": -0.119225,
    "p90_coverage": 90.692781,
}

# MovingAvg30 baseline — from Forecast Accuracy PDF
baseline_ref = {
    "forecast_method": "MovingAvg30 (baseline)",
    "mape": 45.98,
    "wape": 28.89,
    "mae":  None,
    "rmse": None,
    "bias": None,
    "p90_coverage": 32.7,
}

summary = pd.DataFrame([baseline_ref, v1_ref, m_v2])
print("\n=== MODEL COMPARISON ===")
print(summary[["forecast_method", "mape", "wape", "bias", "p90_coverage"]].to_string(index=False))

print("\n=== Beverages / Snacks WAPE (Category Guard) ===")
print(bs_v2.to_string(index=False))

# Champion/Challenger decision (exact from Cell [221])
promote = (
    m_v2["wape"] < v1_ref["wape"] and
    88 <= m_v2["p90_coverage"] <= 92
)
print(f"\n=== DECISION ===")
print(" PROMOTE DemandSense_v2" if promote else "⚠️  KEEP DemandSense_v1 — check metrics")

# Improvement vs MovingAvg30 baseline
wape_improve = (28.89 - m_v2["wape"]) / 28.89 * 100
p90_improve  = m_v2["p90_coverage"] - 32.7
print(f"\n  vs MovingAvg30 baseline:")
print(f"  WAPE improvement  : {wape_improve:.2f}%")
print(f"  P90 coverage gain : {p90_improve:.2f} pts")

# Save summary
# Save summary
summary.to_csv(PROCESSED / "demandSense_model_summary.csv", index=False)
pred_all.to_csv(PROCESSED / "demandSense_v2_predictions.csv", index=False)

# ── NEW: Handoff to Nexus Pipeline ────────────────────────────────
print("\n[Handoff] Formatting predictions for Nexus Pipeline...")
# 1. Create a copy so we don't break Cell 8 and 12
nexus_df = pred_all.copy()

# 2. Rename 'date' to 'forecast_date' (the other columns are already correct!)
nexus_df = nexus_df.rename(columns={'date': 'forecast_date'})

# 3. Keep only the columns Nexus expects
nexus_columns = ['store_id', 'sku_id', 'forecast_date', 'forecast_units', 'lower_bound_90', 'upper_bound_90']
nexus_df = nexus_df[nexus_columns]

# 4. Save directly to your raw CSV folder using the CSV_PATH variable from Cell 1
nexus_target_path = CSV_PATH / 'demand_forecasts.csv'
nexus_df.to_csv(nexus_target_path, index=False)

print(f"   Handoff complete: Forecasts saved directly to {nexus_target_path}")
# ─────────────────────────────────────────────────────────────────

print("\n Evaluation complete")

[Evaluation] Computing metrics on Q4 test set...

=== MODEL COMPARISON ===
       forecast_method      mape      wape      bias  p90_coverage
MovingAvg30 (baseline) 45.980000 28.890000       NaN     32.700000
        DemandSense_v1 40.530127 24.176468 -0.119225     90.692781
        DemandSense_v2 39.900508 24.055682 -0.148075     89.750810

=== Beverages / Snacks WAPE (Category Guard) ===
         model  category      wape
DemandSense_v2 Beverages 24.132614
DemandSense_v2    Snacks 23.967589

=== DECISION ===
 PROMOTE DemandSense_v2

  vs MovingAvg30 baseline:
  WAPE improvement  : 16.73%
  P90 coverage gain : 57.05 pts

 Evaluation complete


In [23]:
# Segment diagnostics — from Forecast.ipynb Cell [110]/[218]
pred_diag = pred_all.copy()
pred_diag["abs_error"] = (pred_diag["target"] - pred_diag["forecast_units"]).abs()
pred_diag["in_90"]     = (
    (pred_diag["target"] >= pred_diag["lower_bound_90"]) &
    (pred_diag["target"] <= pred_diag["upper_bound_90"])
)

def seg_metrics(df, group_col):
    return (df.groupby(group_col, observed=True)
              .apply(lambda g: pd.Series({
                  "wape": g["abs_error"].sum() / g["target"].sum() * 100,
                  "p90":  g["in_90"].mean() * 100,
                  "volume_share": g["target"].sum(),
              }))
              .assign(volume_share=lambda d: d["volume_share"] / d["volume_share"].sum() * 100)
              .reset_index())

diag_tier     = seg_metrics(pred_diag, "foot_traffic_tier")
diag_category = seg_metrics(pred_diag, "category")

# Volume bucket (Q1=lowest demand, Q4=highest)
pred_diag["vol_bucket"] = pd.qcut(
    pred_diag.groupby(["store_id","sku_id"])["target"].transform("mean"),
    q=4, labels=["Q1","Q2","Q3","Q4"]
)
diag_volume = seg_metrics(pred_diag, "vol_bucket")

diag_tier.to_csv(PROCESSED / "diag_kpi_by_tier.csv", index=False)
diag_category.to_csv(PROCESSED / "diag_kpi_by_category.csv", index=False)
diag_volume.to_csv(PROCESSED / "diag_kpi_by_volume_bucket.csv", index=False)
print("Segment diagnostics saved")

Segment diagnostics saved


In [24]:
# ─────────────────────────────────────────────────────────────────
# CELL 8 — Safety stock + replenishment inputs
# Base formula from Forecast.ipynb Cell [226]
# Enhancement: tiered CSL by store foot_traffic_tier
# ─────────────────────────────────────────────────────────────────
print("[Replenishment] Computing tiered safety stock...")

pred = pred_all.copy()

# Forecast moments from quantile models (exact from Cell [226])
pred["mu_daily"]    = pred["forecast_units"].clip(lower=0)
pred["sigma_daily"] = (
    (pred["upper_bound_90"] - pred["forecast_units"]) / 1.645
).clip(lower=0)

# Tiered z-score (NEW — replaces single CSL_NEW = 0.90)
pred["csl"]   = pred["foot_traffic_tier"].map(CSL_BY_TIER).fillna(0.95)
pred["z_val"] = pred["csl"].map(Z_MAP)

# Old policy (no safety stock — baseline)
pred["target_upto_old"] = np.ceil(pred["mu_daily"])

# New policy (tiered safety stock)
pred["target_upto_new"] = np.ceil(
    pred["mu_daily"] + pred["z_val"] * pred["sigma_daily"]
)

# Merge supplier lead time
sku_sup = products[["sku_id", "supplier_id"]].merge(
    suppliers[["supplier_id", "lead_time_days_avg",
               "lead_time_days_std", "reliability_score"]],
    on="supplier_id", how="left"
)
pred = pred.merge(sku_sup, on="sku_id", how="left")
pred["lead_time_days_avg"] = pred["lead_time_days_avg"].fillna(7)
pred["lead_time_days_std"] = pred["lead_time_days_std"].fillna(2)

# Safety stock and ROP with lead time
pred["safety_stock_units"] = np.ceil(
    pred["z_val"] * pred["sigma_daily"] * np.sqrt(pred["lead_time_days_avg"])
)
pred["reorder_point"] = np.ceil(
    pred["mu_daily"] * pred["lead_time_days_avg"] + pred["safety_stock_units"]
)

repl_cols = [
    "store_id", "sku_id", "date", "foot_traffic_tier", "category",
    "target", "forecast_units", "lower_bound_90", "upper_bound_90",
    "mu_daily", "sigma_daily", "csl", "z_val",
    "target_upto_old", "target_upto_new",
    "safety_stock_units", "reorder_point",
    "lead_time_days_avg", "lead_time_days_std", "reliability_score",
]
repl_inputs = pred[[c for c in repl_cols if c in pred.columns]].copy()
repl_inputs.to_csv(PROCESSED / "replenishment_policy_inputs_demandsense.csv", index=False)

print(f"  Replenishment inputs: {len(repl_inputs):,} rows")
print(f"\n  Average safety stock by tier:")
print(
    repl_inputs.groupby("foot_traffic_tier")
    .agg(avg_ss=("safety_stock_units", "mean"),
         avg_z=("z_val", "first"))
    .round(2)
)
print("\n Safety stock complete")


 #Per-store service level from test period (for Streamlit)
store_sl = (
    pred_all.groupby("store_id")
    .apply(lambda g: pd.Series({
        "total_demand":    g["target"].sum(),
        "est_lost_units":  (g["target"] - g["forecast_units"].clip(upper=g["target"])).clip(lower=0).sum(),
    }))
    .assign(service_level=lambda d: (1 - d["est_lost_units"] / d["total_demand"]) * 100)
    .reset_index()
)
store_sl.to_csv(PROCESSED / "store_service_levels.csv", index=False)

[Replenishment] Computing tiered safety stock...
  Replenishment inputs: 3,271,439 rows

  Average safety stock by tier:
                   avg_ss  avg_z
foot_traffic_tier               
High                24.63   1.96
Low                 13.96   1.64
Medium              17.53   1.64
Premium             27.99   1.96

 Safety stock complete


In [25]:
# ─────────────────────────────────────────────────────────────────
# CELL 9 — Backtest simulation
# Exact from Forecast.ipynb Cell [228]
# Tests Black Friday + Christmas weeks
# ─────────────────────────────────────────────────────────────────
from collections import defaultdict

print("[Backtest] Running policy simulation for Black Friday + Christmas...")

df_bt = repl_inputs.sort_values(["store_id", "sku_id", "date"]).copy()
df_bt["week"] = df_bt["date"].dt.to_period("W").astype(str)

weeks_to_show = ["2025-11-24/2025-11-30", "2025-12-22/2025-12-28"]
demo = df_bt[df_bt["week"].isin(weeks_to_show)].copy()

LEAD_TIME_DAYS    = 2
REVIEW_PERIOD_DAYS = 1
effective_h = 1.0 + (LEAD_TIME_DAYS / REVIEW_PERIOD_DAYS)
demo["init_on_hand"] = np.ceil(
    demo["mu_daily"] * (LEAD_TIME_DAYS + REVIEW_PERIOD_DAYS)
)
CAP_MULTIPLIER = 1.8

def simulate_policy(data, policy_name):
    """Exact from Forecast.ipynb Cell [228]."""
    in_transit = defaultdict(list)
    on_hand = {}
    out = []

    for r in data.itertuples(index=False):
        key = (r.store_id, r.sku_id)
        dt  = r.date

        if key not in on_hand:
            on_hand[key] = float(r.init_on_hand)

        arrivals, remaining = 0.0, []
        for arr_dt, qty in in_transit[key]:
            if arr_dt <= dt:
                arrivals += qty
            else:
                remaining.append((arr_dt, qty))
        in_transit[key] = remaining
        on_hand[key] += arrivals

        on_order_qty  = sum(q for _, q in in_transit[key])
        inv_position  = on_hand[key] + on_order_qty

        base_h  = r.mu_daily * effective_h
        sigma_h = r.sigma_daily * np.sqrt(effective_h)

        if policy_name == "OLD":
            raw_target = float(np.ceil(base_h))
        else:
            # Use row-level z_val for tiered CSL
            z = float(r.z_val)
            raw_target = float(np.ceil(base_h + z * sigma_h))

        cap_target = float(np.ceil(r.upper_bound_90 * CAP_MULTIPLIER))
        target_upto = min(raw_target, cap_target)

        order_qty = 0.0
        if (dt.toordinal() % REVIEW_PERIOD_DAYS == 0) and (inv_position < target_upto):
            order_qty = target_upto - inv_position
            in_transit[key].append(
                (dt + pd.Timedelta(days=LEAD_TIME_DAYS), order_qty)
            )

        demand    = float(r.target)
        sales     = min(on_hand[key], demand)
        lost_sales = max(0.0, demand - on_hand[key])
        on_hand[key] = max(0.0, on_hand[key] - demand)

        out.append({
            "policy": policy_name, "store_id": r.store_id,
            "sku_id": r.sku_id, "date": dt, "week": r.week,
            "target": demand, "sales": sales, "lost_sales": lost_sales,
            "stockout_flag": lost_sales > 0, "order_qty": order_qty,
            "on_hand_end": on_hand[key],
        })
    return pd.DataFrame(out)

old_res = simulate_policy(demo, "OLD")
new_res = simulate_policy(demo, "NEW")
res = pd.concat([old_res, new_res], ignore_index=True)

ov = (
    res.groupby("policy", as_index=False)
    .agg(
        stockout_events=("stockout_flag", "sum"),
        lost_units=("lost_sales", "sum"),
        demand_units=("target", "sum"),
    )
)
ov["service_level_%"] = (
    1 - ov["lost_units"] / ov["demand_units"].replace(0, np.nan)
) * 100

print("\n=== OVERALL POLICY COMPARISON (Black Friday + Christmas) ===")
print(ov.to_string(index=False))

old = ov[ov["policy"] == "OLD"].iloc[0]
new = ov[ov["policy"] == "NEW"].iloc[0]
print(f"\n  Stockout reduction : {(old['stockout_events']-new['stockout_events'])/old['stockout_events']*100:.2f}%")
print(f"  Lost units reduction: {(old['lost_units']-new['lost_units'])/old['lost_units']*100:.2f}%")

res.to_csv(PROCESSED / "backtest_daily_demandsense.csv", index=False)
print("\n Backtest complete")

[Backtest] Running policy simulation for Black Friday + Christmas...

=== OVERALL POLICY COMPARISON (Black Friday + Christmas) ===
policy  stockout_events  lost_units  demand_units  service_level_%
   NEW           126756    920934.0     6079523.0        84.851871
   OLD           138027    960826.0     6079523.0        84.195701

  Stockout reduction : 8.17%
  Lost units reduction: 4.15%

 Backtest complete


In [ ]:
# Weekly monitoring — exact from Forecast.ipynb Cell [224]
pred_mon = pred_all.copy()
pred_mon["error"]     = pred_mon["target"] - pred_mon["forecast_units"]
pred_mon["abs_error"] = pred_mon["error"].abs()
pred_mon["ape"]       = pred_mon["abs_error"] / pred_mon["target"].replace(0, np.nan) * 100
pred_mon["in_90"]     = (
    (pred_mon["target"] >= pred_mon["lower_bound_90"]) &
    (pred_mon["target"] <= pred_mon["upper_bound_90"])
)
pred_mon["week"] = pred_mon["date"].dt.to_period("W").astype(str)

weekly_monitor = (
    pred_mon.groupby("week", as_index=False)
    .agg(
        mape         = ("ape",      "mean"),
        abs_err_sum  = ("abs_error","sum"),
        actual_sum   = ("target",   "sum"),
        bias         = ("error",    "mean"),
        p90_coverage = ("in_90",    "mean"),
    )
)
weekly_monitor["wape"]        = weekly_monitor["abs_err_sum"] / weekly_monitor["actual_sum"] * 100
weekly_monitor["p90_coverage"] = weekly_monitor["p90_coverage"] * 100
weekly_monitor = weekly_monitor[["week","mape","wape","bias","p90_coverage"]]
weekly_monitor.to_csv(PROCESSED / "weekly_monitor_demandsense.csv", index=False)
print(weekly_monitor.tail(12).to_string(index=False))

                 week      mape      wape      bias  p90_coverage
2025-10-13/2025-10-19 39.985515 23.709822  0.001207     89.627447
2025-10-20/2025-10-26 40.115397 23.780621 -0.018458     89.556712
2025-10-27/2025-11-02 39.799720 23.377302 -0.015826     90.036233
2025-11-03/2025-11-09 38.227128 22.669049  0.110704     90.117338
2025-11-10/2025-11-16 38.107816 22.633569  0.062111     89.910050
2025-11-17/2025-11-23 38.446289 22.718606 -0.050457     89.571736
2025-11-24/2025-11-30 50.424358 32.321044 -1.718879     90.040120
2025-12-01/2025-12-07 34.429350 21.679184  0.766958     89.120801
2025-12-08/2025-12-14 35.263431 21.509522  0.479843     89.163215
2025-12-15/2025-12-21 35.569270 21.400517  0.364174     89.143254
2025-12-22/2025-12-28 45.610448 27.944014 -1.245441     89.946015
2025-12-29/2026-01-04 48.267976 29.294926 -1.823146     91.165737


: 

In [2]:
# ─────────────────────────────────────────────────────────────────
# RECOVERY CELL — Run this after kernel crash
# Loads only what Cell 10 and Cell 11 need
# Skips re-running the 90-minute training pipeline
# ─────────────────────────────────────────────────────────────────
import gc

print("[Recovery] Loading saved outputs from completed cells...")

# Load stores/products/suppliers (needed for Cell 10 feature join)
stores    = pd.read_csv(CSV_PATH / "stores.csv")
products  = pd.read_csv(CSV_PATH / "products.csv")
suppliers = pd.read_csv(CSV_PATH / "suppliers.csv")
print(f"  stores: {len(stores):,}  products: {len(products):,}  suppliers: {len(suppliers):,}")

# Load saved predictions (output of Cell 6) — skip retraining
print("[Recovery] Loading saved predictions...")
pred_all = pd.read_csv(
    PROCESSED / "demandSense_v2_predictions.csv",
    parse_dates=["date"]
)
print(f"  pred_all: {pred_all.shape}")

# Rebuild train/valid splits from train_master_full.csv
# (needed by Cell 11 to retrain and save model objects)
print("[Recovery] Rebuilding train/valid splits (needed for Cell 11 save)...")
print("  Loading train_master_full.csv...")
df_raw = pd.read_csv(NB_DIR / "train_master_full.csv", parse_dates=["date"])
if "actual_units" in df_raw.columns:
    df_raw = df_raw.drop(columns=["actual_units"])

# Add only the stockout proxy (small join — skip promo expand to save memory)
stockouts = pd.read_csv(CSV_PATH / "stockout_events.csv", parse_dates=["stockout_date"])
lost = (
    stockouts
    .groupby(["store_id","sku_id","stockout_date"], as_index=False)["estimated_lost_units"]
    .sum()
    .rename(columns={"stockout_date":"date","estimated_lost_units":"lost_units_proxy"})
)
df_raw = df_raw.merge(lost, on=["store_id","sku_id","date"], how="left")
df_raw["lost_units_proxy"] = df_raw["lost_units_proxy"].fillna(0.0).clip(
    0, df_raw["lost_units_proxy"].quantile(0.99)
)
df_raw["stockout_flag"] = (df_raw["lost_units_proxy"] > 0).astype(int)

# Add promo features as zeros for split rebuild
# (we already have correct predictions saved — these splits are for Cell 11 retrain only)
for col in ["discount_pct","demand_lift_factor","promo_depth_x_flag","promo_lift_x_flag"]:
    if col not in df_raw.columns:
        df_raw[col] = 0.0

del lost, stockouts
gc.collect()

# Rebuild splits
train = df_raw[df_raw["date"] <= TRAIN_END].copy()
valid = df_raw[(df_raw["date"] >= VALID_START) & (df_raw["date"] <= VALID_END)].copy()
test  = df_raw[(df_raw["date"] >= TEST_START)  & (df_raw["date"] <= TEST_END)].copy()

for c in CAT_COLS:
    if c in train.columns:
        train[c] = train[c].astype("category")
        valid[c] = valid[c].astype("category")
        test[c]  = test[c].astype("category")

for split in [train, valid, test]:
    for col in FORECAST_FEATURES:
        if col in split.columns and col not in CAT_COLS:
            split[col] = split[col].fillna(0)

del df_raw
gc.collect()

print(f"  train: {len(train):,}  valid: {len(valid):,}  test: {len(test):,}")
print("\n✅ Recovery complete — ready to run Cell 10 then Cell 11")

[Recovery] Loading saved outputs from completed cells...
  stores: 478  products: 1,265  suppliers: 85
[Recovery] Loading saved predictions...
  pred_all: (3271439, 10)
[Recovery] Rebuilding train/valid splits (needed for Cell 11 save)...
  Loading train_master_full.csv...
  train: 18,355,832  valid: 3,271,115  test: 3,271,439

✅ Recovery complete — ready to run Cell 10 then Cell 11


In [3]:
# ── Free memory before loading raw CSVs for phantom model ─────────
# pred_all alone is 3.27M × many columns — free what we can
import gc
gc.collect()
print(f"  Starting phantom cell...")
# rest of Cell 10 continues unchanged from here

  Starting phantom cell...


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 10 — Phantom inventory detection
# Logic from Phantom_Prevention_new.ipynb — all 5 fixes applied
# Uses raw inventory_snapshots + sales_transactions
# Separate from forecast pipeline — different feature set
# ─────────────────────────────────────────────────────────────────
import duckdb

print("[Phantom] Building daily base table from raw CSVs...")

# Use DuckDB for memory-safe aggregation
con = duckdb.connect()

daily_sales_ph = con.execute(f"""
    SELECT
        store_id, sku_id,
        CAST(sale_date AS DATE) AS sale_date,
        SUM(units_sold) AS units_sold
    FROM read_csv_auto('{CSV_PATH}/sales_transactions.csv')
    GROUP BY store_id, sku_id, CAST(sale_date AS DATE)
    ORDER BY store_id, sku_id, sale_date
""").df()
daily_sales_ph["sale_date"] = pd.to_datetime(daily_sales_ph["sale_date"])

inventory_raw = con.execute(f"""
    SELECT store_id, sku_id,
           CAST(snapshot_date AS DATE) AS snapshot_date,
           units_on_hand, units_in_backroom, days_of_supply
    FROM read_csv_auto('{CSV_PATH}/inventory_snapshots.csv')
    ORDER BY store_id, sku_id, snapshot_date
""").df()
inventory_raw["snapshot_date"] = pd.to_datetime(inventory_raw["snapshot_date"])
con.close()

print(f"  daily_sales: {len(daily_sales_ph):,} rows")
print(f"  inventory:   {len(inventory_raw):,} rows")

# Forward-fill inventory (from retail_eda.ipynb Cell [11])
# Forward-fill inventory — fixed for pandas 2.x
print("[Phantom] Forward-filling inventory to daily frequency...")

def ffill_group(g):
    return g.resample("D").ffill()

inv_daily = (
    inventory_raw
    .set_index("snapshot_date")
    .groupby(["store_id", "sku_id"], group_keys=False)  # ← key fix: group_keys=False
    .apply(ffill_group)
    .reset_index()
    .rename(columns={"snapshot_date": "sale_date"})
)
inv_daily["units_total"] = inv_daily["units_on_hand"] + inv_daily["units_in_backroom"]

print(f"  inv_daily shape: {inv_daily.shape}")
print(f"  Columns: {list(inv_daily.columns)}")

# Join inventory + sales
base_ph = inv_daily[[
    "store_id", "sku_id", "sale_date",
    "units_on_hand", "units_in_backroom", "units_total", "days_of_supply"
]].merge(daily_sales_ph, on=["store_id", "sku_id", "sale_date"], how="left")
base_ph["units_sold"] = base_ph["units_sold"].fillna(0)
base_ph = base_ph.sort_values(["store_id", "sku_id", "sale_date"]).reset_index(drop=True)
print(f"  Base table: {len(base_ph):,} rows")

# Rolling + lag features (vectorized — no Python loop)
print("[Phantom] Computing rolling and lag features...")

def rolling_feats_phantom(g):
    s = g["units_sold"]
    g = g.copy()
    g["rolling_7d_avg"]  = s.shift(1).rolling(7,  min_periods=1).mean()
    g["rolling_14d_avg"] = s.shift(1).rolling(14, min_periods=1).mean()
    g["rolling_30d_avg"] = s.shift(1).rolling(30, min_periods=3).mean()
    g["rolling_7d_std"]  = s.shift(1).rolling(7,  min_periods=2).std().fillna(0)
    g["lag_1"]  = s.shift(1)
    g["lag_7"]  = s.shift(7)
    g["lag_14"] = s.shift(14)
    g["demand_drop"] = (g["rolling_7d_avg"] - g["rolling_14d_avg"]).clip(upper=0).abs()

    # Vectorized zero streak — no Python loop
    is_zero  = (s == 0).astype(int)
    group_id = (is_zero != is_zero.shift()).cumsum()
    g["zero_streak"] = is_zero.groupby(group_id).cumcount() + 1
    g["zero_streak"] = g["zero_streak"].where(is_zero == 1, 0)

    g["avg_before_zero"] = np.where(g["zero_streak"] > 0, g["rolling_14d_avg"], 0)
    g["demand_ratio"]    = g["rolling_7d_avg"] / (g["rolling_30d_avg"] + 1e-5)
    return g

base_ph = (
    base_ph
    .groupby(["store_id", "sku_id"], group_keys=False)
    .apply(rolling_feats_phantom)
    .reset_index(drop=True)
)

for col in ["rolling_7d_avg","rolling_14d_avg","rolling_30d_avg",
            "lag_1","lag_7","lag_14","rolling_7d_std","demand_drop"]:
    base_ph[col] = base_ph[col].fillna(0)

# 3 phantom rules (from Phantom_Prevention_new.ipynb — all 3 enabled)
demand_threshold_25 = (
    base_ph.groupby("sku_id")["rolling_7d_avg"]
    .transform(lambda x: x.quantile(0.25))
)
base_ph["rule_zero_streak"]  = ((base_ph["zero_streak"] >= 3) & (base_ph["units_total"] > 0)).astype(int)
base_ph["rule_dos_mismatch"] = (
    (base_ph["days_of_supply"] > 0) &
    (base_ph["units_sold"] == 0) &
    (base_ph["rolling_7d_avg"] > demand_threshold_25)
).astype(int)
base_ph["rule_flatline"] = ((base_ph["units_total"] > 0) & (base_ph["demand_ratio"] < 0.2)).astype(int)
base_ph["rule_phantom_flag"] = (
    (base_ph["rule_zero_streak"] == 1) |
    (base_ph["rule_dos_mismatch"] == 1) |
    (base_ph["rule_flatline"]     == 1)
).astype(int)

# Store + product features with LabelEncoder
le_region = LabelEncoder(); le_format = LabelEncoder()
le_tier   = LabelEncoder(); le_cat    = LabelEncoder()

stores_fe = stores[["store_id","region","store_format","foot_traffic_tier","sq_footage"]].copy()
stores_fe["region_code"]       = le_region.fit_transform(stores_fe["region"])
stores_fe["store_format_code"] = le_format.fit_transform(stores_fe["store_format"])
stores_fe["foot_traffic_code"] = le_tier.fit_transform(stores_fe["foot_traffic_tier"])

products_fe = products[["sku_id","category","unit_price","is_perishable","reorder_point","safety_stock"]].copy()
products_fe["category_code"] = le_cat.fit_transform(products_fe["category"])
products_fe["is_perishable"]  = products_fe["is_perishable"].astype(int)

base_ph = base_ph.merge(
    stores_fe[["store_id","region_code","store_format_code","foot_traffic_code","sq_footage"]],
    on="store_id", how="left"
)
base_ph = base_ph.merge(
    products_fe[["sku_id","category_code","unit_price","is_perishable","reorder_point","safety_stock"]],
    on="sku_id", how="left"
)

base_ph["day_of_week"] = base_ph["sale_date"].dt.dayofweek
base_ph["month"]       = base_ph["sale_date"].dt.month
base_ph["is_weekend"]  = (base_ph["sale_date"].dt.dayofweek >= 5).astype(int)

# Phantom label (Fix 1 from Phantom_Prevention_new.ipynb)
demand_threshold_30 = (
    base_ph.groupby("sku_id")["rolling_14d_avg"]
    .transform(lambda x: x.quantile(0.30))
)
base_ph["is_phantom_label"] = (
    (base_ph["zero_streak"] >= 3) &
    (base_ph["units_total"] > 0) &
    (base_ph["rolling_14d_avg"] > demand_threshold_30)
).astype(int)

PHANTOM_FEATURES = [
    # Inventory signals
    "units_on_hand", "units_in_backroom", "days_of_supply",

    # Demand history (NO zero_streak — would cause leakage with label)
    "rolling_7d_avg", "rolling_7d_std",
    "lag_1", "lag_7", "lag_14", "demand_drop",

    # Store context
    "region_code", "store_format_code", "foot_traffic_code", "sq_footage",

    # Product context
    "category_code", "unit_price", "is_perishable", "reorder_point", "safety_stock",

    # Calendar
    "day_of_week", "month", "is_weekend",
]
PHANTOM_FEATURES = [c for c in PHANTOM_FEATURES if c in base_ph.columns]

# Time split for phantom
PHANTOM_CUTOFF = "2025-09-30"   # exact from Phantom_Prevention_new.ipynb
ph_train = base_ph[(base_ph["sale_date"] <= PHANTOM_CUTOFF) & (base_ph["units_total"] > 0)].copy()
ph_test  = base_ph[(base_ph["sale_date"] >  PHANTOM_CUTOFF) & (base_ph["units_total"] > 0)].copy()

X_ph_tr = ph_train[PHANTOM_FEATURES].fillna(0)
y_ph_tr = ph_train["is_phantom_label"]
X_ph_te = ph_test[PHANTOM_FEATURES].fillna(0)
y_ph_te = ph_test["is_phantom_label"]

print(f"\n  Phantom train: {len(X_ph_tr):,} rows, "
      f"{y_ph_tr.sum():,} positives ({y_ph_tr.mean()*100:.2f}%)")

# Train RF (from Phantom_Prevention_new.ipynb)
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=15, min_samples_leaf=20,
    max_features="sqrt", class_weight="balanced",
    random_state=42, n_jobs=-1
)
rf_model.fit(X_ph_tr, y_ph_tr)

# F2-optimized threshold (Fix 3 from Phantom_Prevention_new.ipynb)
phantom_proba = rf_model.predict_proba(X_ph_te)[:, 1]
prec_vals, rec_vals, thresh_vals = precision_recall_curve(y_ph_te, phantom_proba)
f2_scores = (5 * prec_vals * rec_vals) / (4 * prec_vals + rec_vals + 1e-8)
best_idx  = np.argmax(f2_scores[:-1])
best_threshold = thresh_vals[best_idx]

y_pred_ph = (phantom_proba >= best_threshold).astype(int)
print(f"\n  Optimal threshold : {best_threshold:.3f}")
print(f"  ROC-AUC           : {roc_auc_score(y_ph_te, phantom_proba):.4f}")
print(f"\n{classification_report(y_ph_te, y_pred_ph, target_names=['Normal','Phantom'])}")
print(" Phantom model trained")

[Phantom] Building daily base table from raw CSVs...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  daily_sales: 25,969,106 rows
  inventory:   4,015,200 rows
[Phantom] Forward-filling inventory to daily frequency...
  inv_daily shape: (27876960, 7)
  Columns: ['sale_date', 'store_id', 'sku_id', 'units_on_hand', 'units_in_backroom', 'days_of_supply', 'units_total']
  Base table: 27,876,960 rows
[Phantom] Computing rolling and lag features...

  Phantom train: 23,512,909 rows, 64,004 positives (0.27%)

  Optimal threshold : 0.908
  ROC-AUC           : 0.9994

              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00   3269149
     Phantom       0.63      0.97      0.77     11191

    accuracy                           1.00   3280340
   macro avg       0.82      0.99      0.88   3280340
weighted avg       1.00      1.00      1.00   3280340

 Phantom model trained


In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 11 — Save all models
# FIX: corrected indentation of m.fit() and print inside function
# ─────────────────────────────────────────────────────────────────
print("[Save] Saving all models and artifacts...")
print("  Re-training to get saveable model objects...")

def train_model_object(tr, va, objective, alpha=None, tag=""):
    params = dict(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        min_child_samples=300, subsample=0.8, colsample_bytree=0.8,
        force_col_wise=True, n_jobs=-1, random_state=42, verbose=-1
    )
    if objective == "quantile":
        params["objective"] = "quantile"
        params["alpha"]     = alpha
        params["metric"]    = "quantile"
    else:
        params["objective"] = "regression"
        params["metric"]    = "l1"

    m = LGBMRegressor(**params)
    for c in CAT_COLS:
        tr[c] = tr[c].astype("category")
        va[c] = va[c].astype("category")

    # ── Correct indentation — all lines inside function ──────────
    m.fit(
        tr[FORECAST_FEATURES], tr["target"],
        eval_set=[(va[FORECAST_FEATURES], va["target"])],
        eval_metric=params["metric"],
        categorical_feature=CAT_COLS,
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(200),
        ]
    )
    print(f"    {tag} — best iter: {m.best_iteration_}")
    return m


tr_hi = train[train["foot_traffic_tier"].isin(HI_TIERS)].copy()
va_hi = valid[valid["foot_traffic_tier"].isin(HI_TIERS)].copy()
tr_lo = train[train["foot_traffic_tier"].isin(LO_TIERS)].copy()
va_lo = valid[valid["foot_traffic_tier"].isin(LO_TIERS)].copy()

p50_hi  = train_model_object(tr_hi.copy(), va_hi.copy(), "regression",          tag="P50 High/Premium")
p50_lo  = train_model_object(tr_lo.copy(), va_lo.copy(), "regression",          tag="P50 Low/Medium")
p05_all = train_model_object(train.copy(), valid.copy(), "quantile", alpha=0.05, tag="P05")
p95_all = train_model_object(train.copy(), valid.copy(), "quantile", alpha=0.95, tag="P95")

# Save everything
print("\n  Writing .joblib files...")
joblib.dump(p50_hi,            MODELS_DIR / "demandsense_p50_high.joblib")
joblib.dump(p50_lo,            MODELS_DIR / "demandsense_p50_low.joblib")
joblib.dump(p05_all,           MODELS_DIR / "demandsense_p05.joblib")
joblib.dump(p95_all,           MODELS_DIR / "demandsense_p95.joblib")
joblib.dump(rf_model,          MODELS_DIR / "rf_phantom.joblib")
joblib.dump(FORECAST_FEATURES, MODELS_DIR / "forecast_features.joblib")
joblib.dump(PHANTOM_FEATURES,  MODELS_DIR / "phantom_features.joblib")
joblib.dump(best_threshold,    MODELS_DIR / "phantom_threshold.joblib")
joblib.dump(HI_TIERS,          MODELS_DIR / "hi_tiers.joblib")
joblib.dump(LO_TIERS,          MODELS_DIR / "lo_tiers.joblib")
joblib.dump(CAT_COLS,          MODELS_DIR / "cat_cols.joblib")
joblib.dump(CSL_BY_TIER,       MODELS_DIR / "csl_by_tier.joblib")
joblib.dump(le_region,         MODELS_DIR / "le_region.joblib")
joblib.dump(le_format,         MODELS_DIR / "le_format.joblib")
joblib.dump(le_tier,           MODELS_DIR / "le_tier.joblib")
joblib.dump(le_cat,            MODELS_DIR / "le_cat.joblib")
stores_fe.to_parquet(   MODELS_DIR / "stores_features.parquet",   index=False)
products_fe.to_parquet( MODELS_DIR / "products_features.parquet", index=False)

saved = sorted(MODELS_DIR.glob("*"))
print(f"\n  Saved {len(saved)} files:")
for f in saved:
    print(f"    {f.name}")
print("\n All artifacts saved — ready for Ollama LLM integration")

[Save] Saving all models and artifacts...
  Re-training to get saveable model objects...
[200]	valid_0's l1: 3.0072
[400]	valid_0's l1: 3.00291
    P50 High/Premium — best iter: 499
[200]	valid_0's l1: 2.00546
[400]	valid_0's l1: 2.00422
    P50 Low/Medium — best iter: 410
[200]	valid_0's quantile: 0.286028
[400]	valid_0's quantile: 0.285598
    P05 — best iter: 500
[200]	valid_0's quantile: 0.351069
[400]	valid_0's quantile: 0.350017
    P95 — best iter: 500

  Writing .joblib files...

  Saved 18 files:
    cat_cols.joblib
    csl_by_tier.joblib
    demandsense_p05.joblib
    demandsense_p50_high.joblib
    demandsense_p50_low.joblib
    demandsense_p95.joblib
    forecast_features.joblib
    hi_tiers.joblib
    le_cat.joblib
    le_format.joblib
    le_region.joblib
    le_tier.joblib
    lo_tiers.joblib
    phantom_features.joblib
    phantom_threshold.joblib
    products_features.parquet
    rf_phantom.joblib
    stores_features.parquet

 All artifacts saved — ready for Ollama LLM

In [11]:
# ─────────────────────────────────────────────────────────────────
# CELL 12 — Pipeline summary
# All metrics aligned with Forecast Accuracy & Stockout Impact PDF
# ─────────────────────────────────────────────────────────────────

# ── Segment-level WAPE (from PDF Section 6) ───────────────────────
seg_wape = (
    pred_all.copy()
    .assign(abs_err=lambda d: (d["target"] - d["forecast_units"]).abs())
    .groupby("foot_traffic_tier")
    .apply(lambda g: g["abs_err"].sum() / g["target"].sum() * 100)
    .rename("wape_%")
    .sort_values()
)

# ── True demand adjustment (from PDF Section 7) ───────────────────
# True demand = actual sales + estimated lost units
pred_true = pred_all.merge(
    stockouts[["store_id", "sku_id", "stockout_date", "estimated_lost_units"]]
    .rename(columns={"stockout_date": "date"}),
    on=["store_id", "sku_id", "date"], how="left"
)
pred_true["estimated_lost_units"] = pred_true["estimated_lost_units"].fillna(0)
pred_true["true_demand"] = pred_true["target"] + pred_true["estimated_lost_units"]
true_demand_wape = (
    (pred_true["forecast_units"] - pred_true["true_demand"]).abs().sum()
    / pred_true["true_demand"].sum() * 100
)

# ── Impact score by category (from PDF Section 8) ─────────────────
# Impact Score = WAPE × Volume Share
cat_stats = (
    pred_all.copy()
    .assign(abs_err=lambda d: (d["target"] - d["forecast_units"]).abs())
    .groupby("category")
    .agg(actual_sum=("target", "sum"), abs_err_sum=("abs_err", "sum"))
)
total_demand = cat_stats["actual_sum"].sum()
cat_stats["wape_%"]       = cat_stats["abs_err_sum"] / cat_stats["actual_sum"] * 100
cat_stats["volume_share"] = cat_stats["actual_sum"] / total_demand * 100
cat_stats["impact_score"] = cat_stats["wape_%"] * cat_stats["volume_share"]
cat_stats = cat_stats.sort_values("impact_score", ascending=False)

# ── Service level from backtest (from PDF Section 13) ─────────────
try:
    bt = pd.read_csv(PROCESSED / "backtest_daily_demandsense.csv")
    old_bt = bt[bt["policy"] == "OLD"]
    new_bt = bt[bt["policy"] == "NEW"]
    sl_old = (1 - old_bt["lost_sales"].sum() / old_bt["target"].sum()) * 100
    sl_new = (1 - new_bt["lost_sales"].sum() / new_bt["target"].sum()) * 100
    so_old = old_bt["stockout_flag"].sum()
    so_new = new_bt["stockout_flag"].sum()
    so_reduction   = (so_old - so_new) / so_old * 100
    lost_reduction = (old_bt["lost_sales"].sum() - new_bt["lost_sales"].sum()) / old_bt["lost_sales"].sum() * 100
    backtest_available = True
except Exception:
    backtest_available = False

# ── Champion/Challenger decision ──────────────────────────────────
promote = (m_v2["wape"] < 24.18) and (88 <= m_v2["p90_coverage"] <= 92)
decision = " PROMOTE DemandSense_v2" if promote else "⚠️  KEEP DemandSense_v1"

# ── Print full summary ─────────────────────────────────────────────
print("=" * 65)
print("  DemandSense_v2 — PIPELINE COMPLETE")
print("  Retail AI Shelf Optimization | 478 stores · 1,265 SKUs")
print("=" * 65)

print("""
┌─────────────────────────────────────────────────────────────┐
│  FORECAST MODEL PERFORMANCE  (ref: Forecast Accuracy PDF)   │
├──────────────────┬──────────────┬─────────────┬─────────────┤
│  Metric          │  MovingAvg30 │ DemandSense │  Improvement│
│                  │  (baseline)  │    _v2      │             │
├──────────────────┼──────────────┼─────────────┼─────────────┤""")

metrics_rows = [
    ("MAPE  (%)",    45.98,              m_v2["mape"],        "lower better"),
    ("WAPE  (%)",    28.89,              m_v2["wape"],        "PRIMARY metric"),
    ("MAE   (units)", None,             m_v2["mae"],         "unit-level err"),
    ("RMSE  (units)", None,             m_v2["rmse"],        "penalises large"),
    ("Bias  (units)", None,             m_v2["bias"],        "0 = balanced"),
    ("P90 cov (%)",  32.7,              m_v2["p90_coverage"],"target 88–92%"),
]

for name, baseline, model_val, note in metrics_rows:
    if baseline is not None:
        if name.startswith("WAPE") or name.startswith("MAPE"):
            impr = f"-{baseline - model_val:.2f} pts"
        elif name.startswith("P90"):
            impr = f"+{model_val - baseline:.2f} pts"
        else:
            impr = ""
        b_str = f"{baseline:>8.2f}"
    else:
        b_str = f"{'—':>8}"
        impr  = ""
    print(f"│  {name:<16}│  {b_str}    │  {model_val:>8.2f}   │  {impr:<10} │")

print("└──────────────────┴──────────────┴─────────────┴─────────────┘")

print(f"""
  True Demand Adjustment (PDF Section 7):
    Standard WAPE (based on sales)       : {m_v2['wape']:.2f}%
    Rigorous WAPE (true demand adjusted) : {true_demand_wape:.2f}%
    Hidden demand gap                    : +{true_demand_wape - m_v2['wape']:.2f} pts
    (PDF reference: 24.18% → 28.19%)
""")

print("  Segment-Level WAPE by Store Tier (PDF Section 6):")
print("  ┌──────────────┬──────────┬────────────────┐")
print("  │ Store Tier   │  WAPE %  │  CSL applied   │")
print("  ├──────────────┼──────────┼────────────────┤")
tier_order = ["Premium", "High", "Medium", "Low"]
for tier in tier_order:
    if tier in seg_wape.index:
        csl = CSL_BY_TIER.get(tier, 0.95)
        print(f"  │ {tier:<12} │  {seg_wape[tier]:>5.2f}%  │  {csl:.3f}          │")
print("  └──────────────┴──────────┴────────────────┘")
print("  (PDF reference: Premium 20.57% · High 22.93% · Medium 26.89% · Low 32.75%)")

print(f"""
  Phantom Detector (RF + F2 threshold):
    Optimal threshold : {best_threshold:.3f}
    ROC-AUC           : {roc_auc_score(y_ph_te, phantom_proba):.4f}
    Fix 1  : Labels from inventory-sales mismatch (not stockout table)
    Fix 2  : All 3 phantom rules enabled as features
    Fix 3  : PR-curve F2 threshold (recall-weighted, not fixed 0.55)
    Fix 4  : Lag features lag_1 / lag_7 / lag_14 added
""")

print("  Safety Stock — Tiered CSL (PDF Section 12):")
print("  ┌──────────────┬───────┬─────────┬──────────────────────┐")
print("  │ Tier         │  CSL  │  z-val  │  Formula             │")
print("  ├──────────────┼───────┼─────────┼──────────────────────┤")
print("  │ Premium/High │ 97.5% │  1.960  │  μ + 1.96σ√L         │")
print("  │ Medium       │ 95.0% │  1.645  │  μ + 1.645σ√L        │")
print("  │ Low          │ 95.0% │  1.645  │  μ + 1.645σ√L        │")
print("  └──────────────┴───────┴─────────┴──────────────────────┘")
print("  OLD policy: Target = μ  (no safety stock)")
print("  NEW policy: Target = μ + z × σ  (dynamic safety stock)")

if backtest_available:
    print(f"""
  Policy Backtest — Black Friday + Christmas (PDF Section 13):
  ┌──────────────────────┬────────────┬────────────┐
  │ Metric               │ Old Policy │ New Policy │
  ├──────────────────────┼────────────┼────────────┤
  │ Stockout events      │ {so_old:>10,} │ {so_new:>10,} │
  │ Service level        │ {sl_old:>9.2f}% │ {sl_new:>9.2f}% │
  └──────────────────────┴────────────┴────────────┘
    Stockout reduction  : {so_reduction:.2f}%
    Lost units reduction: {lost_reduction:.2f}%
    (PDF reference: 8.33% stockout reduction · 4.41% lost units reduction)""")

print(f"""
  Top 5 Categories by Impact Score = WAPE × Volume Share (PDF Section 8):""")
for cat, row in cat_stats.head(5).iterrows():
    print(f"    {cat:<22} WAPE {row['wape_%']:>5.2f}% × vol {row['volume_share']:>4.1f}% = {row['impact_score']:>6.1f}")
print("  (PDF reference: Beverages highest · Snacks second)")

print(f"""
  Weekly Monitoring Thresholds (PDF MLOps Section):
    Emergency retrain if WAPE worsens by  : > 1.5 pts
    Emergency retrain if P90 goes outside : 88%–92%
    Emergency retrain if bias >           : 0.5 units (high-volume)
    Scheduled retrain cadence             : every 2 weeks

  Champion/Challenger Decision:
    WAPE improved vs V1 ({m_v2['wape']:.2f}% < 24.18%)    : {'✅' if m_v2['wape'] < 24.18 else '❌'}
    P90 in band 88–92% ({m_v2['p90_coverage']:.2f}%)       : {'✅' if 88 <= m_v2['p90_coverage'] <= 92 else '❌'}
    Decision: {decision}

  Artifacts:
    Models   → {MODELS_DIR}
    Outputs  → {PROCESSED}

  Next step: Ollama setup → src/tools.py → Streamlit app
""")
print("=" * 65)

cat_stats.reset_index().to_csv(PROCESSED / "category_impact_scores.csv", index=False)
print("Category impact scores saved")

  DemandSense_v2 — PIPELINE COMPLETE
  Retail AI Shelf Optimization | 478 stores · 1,265 SKUs

┌─────────────────────────────────────────────────────────────┐
│  FORECAST MODEL PERFORMANCE  (ref: Forecast Accuracy PDF)   │
├──────────────────┬──────────────┬─────────────┬─────────────┤
│  Metric          │  MovingAvg30 │ DemandSense │  Improvement│
│                  │  (baseline)  │    _v2      │             │
├──────────────────┼──────────────┼─────────────┼─────────────┤
│  MAPE  (%)       │     45.98    │     39.90   │  -6.08 pts  │
│  WAPE  (%)       │     28.89    │     24.06   │  -4.83 pts  │
│  MAE   (units)   │         —    │      2.78   │             │
│  RMSE  (units)   │         —    │      3.97   │             │
│  Bias  (units)   │         —    │     -0.15   │             │
│  P90 cov (%)     │     32.70    │     89.75   │  +57.05 pts │
└──────────────────┴──────────────┴─────────────┴─────────────┘

  True Demand Adjustment (PDF Section 7):
    Standard WAPE (based on sa